# EDA 심화 3 · 직접 풀어보는 연습문제 정답지

이 노트북은 `EDA_3_HR_Employee_Churn_Practice.ipynb`의 모든 연습문제에 대한 **표준 모범 코드와 결과 해석**을 제공합니다.


In [ ]:
# [실습 환경 준비]
# 이 셀은 HR 재직자 25명의 샘플 데이터를 생성합니다. 셀을 먼저 실행해주세요.
import pandas as pd
import numpy as np

hr_df = pd.DataFrame({
    "enrollee_id": range(1001, 1026),
    "city_dev_index": [0.92, 0.77, 0.62, 0.78, 0.76, 0.92, 0.92, 0.64, 0.92, 0.75,
                       0.55, 0.92, 0.88, 0.62, 0.91, 0.55, 0.69, 0.89, 0.92, 0.62,
                       0.88, 0.92, 0.74, 0.92, 0.62],
    "gender": ["Male", "Male", None, "Male", "Male", "Female", "Male", "Male", "Female", None,
               "Male", "Male", "Male", None, "Male", "Male", "Male", "Female", "Male", "Male",
               "Female", "Male", "Male", "Male", None],
    "experience_years": [15, 9, 5, 10, 11, 8, 12, 3, 7, 6,
                         1, 20, 14, 4, 16, 2, 7, 18, 10, 3,
                         12, 17, 8, 19, 4],
    "company_size": ["50-99", "100-500", None, "50-99", "10000+", "500-999", "100-500", None, "50-99", None,
                     None, "10000+", "1000-4999", None, "100-500", "50-99", None, "10000+", "500-999", None,
                     "100-500", "10000+", "50-99", "500-999", None],
    "training_hours": [36, 19, 83, 52, 108, 24, 18, 46, 123, 32,
                       25, 48, 16, 75, 42, 60, 31, 28, 55, 92,
                       14, 68, 45, 33, 80],
    "target": [0, 0, 1, 0, 0, 0, 0, 1, 0, 1,
               1, 0, 0, 1, 0, 1, 1, 0, 0, 1,
               0, 0, 0, 0, 1] # 1: 이직 희망, 0: 유지
})


## 문제 1. HR 데이터 결측치 현황 및 결측률 계산 [결측 진단]

`hr_df` 데이터프레임의 결측 상태를 진단하세요:
1. 각 컬럼별 결측치 건수(`isna().sum()`)를 구하세요.
2. 전체 행 수(25행) 대비 결측률(%)을 소수점 첫째 자리까지 계산하여 `missing_report` 표로 출력하세요.
3. 가장 결측률이 높은 컬럼을 명시하세요.


In [ ]:
# 문제 1 풀이
missing_counts = hr_df.isna().sum()
missing_rates = (missing_counts / len(hr_df) * 100).round(1)

missing_report = pd.DataFrame({
    "결측치수": missing_counts,
    "결측률(%)": missing_rates
})

print("=== HR 데이터 결측치 진단 보고서 ===")
print(missing_report[missing_report["결측치수"] > 0])
print(f"\n최다 결측 발생 컬럼: {missing_rates.idxmax()} (결측률: {missing_rates.max()}%)")


**결과 해석 / 확인한 점:**  
1. **선택적 무응답의 패턴:** `company_size`(회사 규모)의 결측치가 8건(32.0%)으로 가장 높게 나타났습니다. 이는 프리랜서나 미취업자, 혹은 스타트업 재직자가 규모 입력을 생략하는 경향과 일치합니다.
2. **단순 삭제(Drop) 위험성:** 32%의 결측을 무조건 지울 경우 특정 고용 형태의 표본이 통째로 증발하여 이직 분석 모델에 심각한 편향이 발생함을 확인했습니다.


## 문제 2. 도시 개발 지수(CDI) 구간화에 따른 이직 희망률 분석 [지표 영향도]

도시 발전도(`city_dev_index`)는 이직 결정의 핵심 외적 요인입니다.
1. `pd.cut()`을 사용하여 CDI를 `0.7`을 기준으로 `"저개발 도시 (<0.7)"`와 `"선진 대도시 (>=0.7)"`의 2개 그룹으로 나누어 `city_tier` 컬럼을 생성하세요.
2. 두 그룹별 전체 직원 수와 이직 희망자 수, 이직률(`target` 평균 * 100)을 비교 집계하세요.


In [ ]:
# 문제 2 풀이
# 1. 도시 등급 분류
bins = [0.0, 0.7, 1.0]
labels = ["저개발 도시 (<0.7)", "선진 대도시 (>=0.7)"]
hr_df["city_tier"] = pd.cut(hr_df["city_dev_index"], bins=bins, labels=labels)

# 2. 도시 등급별 이직률 집계
city_churn = hr_df.groupby("city_tier")["target"].agg(
    직원수="count",
    이직희망자수="sum",
    이직률=lambda x: round(x.mean() * 100, 1)
).reset_index()

print("=== 도시 개발 지수(CDI) 수준별 이직률 분석 ===")
print(city_churn.to_string(index=False))


**결과 해석 / 확인한 점:**  
1. **지역적 인프라와 이직 욕구:** 저개발 도시(<0.7) 소속 직원의 이직률은 무려 100.0%(8명 중 8명)에 달한 반면, 선진 대도시(>=0.7) 직원의 이직률은 5.9%(17명 중 1명)에 불과했습니다.
2. **인사 전략 인사이트:** 낙후 지역 지사 근무자들의 인재 유출이 극심하므로 원격 근무(Remote Work) 지원이나 대도시 본사 전보 배치 등의 리텐션 프로그램이 시급함을 실증했습니다.


## 문제 3. 경력 연수(experience_years)와 이직 의향의 역상관성 검증 [경력 분석]

직원의 총 경력 연수(`experience_years`)가 이직 의향에 미치는 영향을 검증하세요:
1. 이직 희망군(`target == 1`)과 잔류군(`target == 0`)의 평균 경력 연수를 각각 계산하세요.
2. 경력 5년 이하인 '주니어' 그룹과 경력 15년 이상인 '시니어' 그룹의 이직률을 비교하세요.


In [ ]:
# 문제 3 풀이
# 1. 타겟별 평균 경력 연수
mean_exp = hr_df.groupby("target")["experience_years"].mean().round(1)
print(f"잔류군 (0) 평균 경력: {mean_exp[0]}년")
print(f"이직 희망군 (1) 평균 경력: {mean_exp[1]}년")

# 2. 주니어 vs 시니어 이직률 비교
junior_group = hr_df[hr_df["experience_years"] <= 5]
senior_group = hr_df[hr_df["experience_years"] >= 15]

print(f"\n주니어 (5년 이하) 이직률: {junior_group['target'].mean() * 100:.1f}% (표본 {len(junior_group)}명)")
print(f"시니어 (15년 이상) 이직률: {senior_group['target'].mean() * 100:.1f}% (표본 {len(senior_group)}명)")


**결과 해석 / 확인한 점:**  
1. **경력 연수와 이직률의 반비례 관계:** 잔류군의 평균 경력은 12.9년인 반면, 이직 희망군의 평균 경력은 3.9년에 불과했습니다.
2. **주니어 층의 높은 이직 욕구:** 주니어(5년 이하)는 100.0%(7명 중 7명)가 이직을 고려하는 반면, 시니어(15년 이상)는 0.0%(6명 중 0명)로 완벽한 직장 안정성을 보여주었습니다.


## 문제 4. 결측 범주를 별도 'Unknown' 클래스로 보존한 회사 규모별 집계 [결측 처리]

`company_size` 컬럼의 결측치를 버리지 않고 하나의 독립 범주로 분석하려고 합니다:
1. `fillna('Unknown')`을 사용하여 결측값을 대체한 파생 컬럼 `company_size_imputed`를 생성하세요.
2. 각 회사 규모별 직원 수와 이직 희망자 수, 이직률(%)을 집계하세요.


In [ ]:
# 문제 4 풀이
# 1. 결측치 Unknown 대체
hr_df["company_size_imputed"] = hr_df["company_size"].fillna("Unknown")

# 2. 회사 규모별 집계
comp_summary = hr_df.groupby("company_size_imputed")["target"].agg(
    직원수="count",
    이직희망자="sum",
    이직률=lambda x: round(x.mean() * 100, 1)
).reset_index()

print("=== 회사 규모별(Unknown 포함) 이직 현황 ===")
print(comp_summary.sort_values(by="직원수", ascending=False).to_string(index=False))


**결과 해석 / 확인한 점:**  
1. **Unknown 집단의 특성 포착:** 결측값을 'Unknown'으로 보존하여 분석한 결과, Unknown 집단 8명 전원(8명, 100.0%)이 이직 희망자로 나타났습니다.
2. **대기업의 잔류 안정성:** 10,000명 이상 대기업(`10000+`)을 포함한 100인 이상 규모 기업 재직자는 모두 이직률 0.0%로 높은 고용 안정성을 보였습니다(`50-99` 규모는 20.0%). 결측을 삭제하지 않고 범주화함으로써 귀중한 인사이트를 확보했습니다.


## 문제 5. 교육 훈련 시간(training_hours)의 이직 억제 효과 검증 [교육 ROI]

기업의 사내 교육 이수 시간(`training_hours`)이 이직률에 미치는 영향을 검증하세요:
1. `training_hours`의 전체 중앙값을 계산하세요.
2. 중앙값 미만인 '소극 이수자' 그룹과 중앙값 이상인 '적극 이수자' 그룹의 이직률을 각각 산출하세요.


In [ ]:
# 문제 5 풀이
median_hours = hr_df["training_hours"].median()
print(f"사내 교육 시간 전체 중앙값: {median_hours}시간")

low_train = hr_df[hr_df["training_hours"] < median_hours]
high_train = hr_df[hr_df["training_hours"] >= median_hours]

print(f"소극 이수군 (< {median_hours}시간) 이직률: {low_train['target'].mean() * 100:.1f}% (표본 {len(low_train)}명)")
print(f"적극 이수군 (>= {median_hours}시간) 이직률: {high_train['target'].mean() * 100:.1f}% (표본 {len(high_train)}명)")


**결과 해석 / 확인한 점:**  
1. **교육 시간의 양면성:** 교육 시간이 중앙값(45시간) 이상인 적극 이수군의 이직률(46.2%)이 소극 이수군(25.0%)보다 오히려 높게 나타나는 현상이 관찰되었습니다.
2. **현상 분석:** 데이터를 심층 분석한 결과, 이직을 결심한 직원들이 스킬 업그레이드를 위해 무료 사내 교육 과정을 집중 이수한 뒤 외부로 전직을 시도하는 경향이 반영되었음을 추론할 수 있습니다.


## 문제 6. 조기 퇴사 초고위험군 식별 및 데이터 기반 인재 방어 전략 [HR 액션 플랜]

앞선 EDA 분석을 종합하여, 인사팀이 즉시 면담 및 리텐션 보너스를 지급해야 할 '초고위험 인재' 선별 규칙을 세웠습니다:
- 규칙: `city_dev_index < 0.70` (저개발 도시) **AND** `experience_years <= 5` (경력 5년 이하 주니어)
1. 위 조건을 만족하는 위험군 인재들의 명단을 추출하세요.
2. 해당 위험군 환산 이직률과 인사팀 권고사항을 출력하세요.


In [ ]:
# 문제 6 풀이
risk_cond = (hr_df["city_dev_index"] < 0.70) & (hr_df["experience_years"] <= 5)
high_churn_risk = hr_df[risk_cond]

print(f"초고위험 타깃 직원 수: {len(high_churn_risk)}명")
print("\n=== 집중 관리 대상 직원 명단 ===")
print(high_churn_risk[["enrollee_id", "city_dev_index", "experience_years", "company_size_imputed", "target"]])

risk_rate = high_churn_risk["target"].mean() * 100
print(f"\n해당 타깃 그룹의 실제 이직 희망률: {risk_rate:.1f}%")
print("인사팀 권고사항: 저개발 도시 근무 5년 이하 주니어 대상 1:1 면담, 주거·근무환경 지원 및 리텐션 보너스·멘토링 프로그램 즉시 도입")

**결과 해석 / 확인한 점:**  
1. **100% 적중률의 리스크 규칙:** 저개발 도시 근무 + 5년 이하 주니어 조건을 만족하는 7명(ID: 1003, 1008, 1011, 1014, 1016, 1020, 1025) 전원이 이직을 준비 중인 것으로 나타나 100.0%의 예측 적중도를 보였습니다.
2. **HR 솔루션 도출:** 지방 지사 주니어 인재들에게 주거 보조비 지원 및 멘토링 결연 등 조기 리텐션 집중 프로그램이 즉각 도입되어야 함을 데이터로 규명했습니다.
